In [1]:
import numpy as np
import pandas as pd
from vcd.reader import TokenKind, tokenize
import io
import scatools as sca

In [2]:
with open('../waves.vcd', 'rb') as f:
    vcd_bytes = f.read()

In [3]:
vcd_bytes = io.BytesIO(vcd_bytes)

In [4]:
tokens = tokenize(vcd_bytes)

In [59]:
signal_info = pd.DataFrame()

# scopes[0] -> Scope Name
# scopes[1] -> Scope Type (begin | fork | function | module | task)
scopes = []

# vars[0] -> Scope Name
# vars[1] -> Var Type
# vars[2] -> Var Size (width)
# vars[3] -> Var ID
# vars[4] -> Var Name
vars = []

# vector_changes = [Timestamp, Var ID, Value]
value_change = []

for token in toks:
    match token.kind:
        case TokenKind.SCOPE:
            curr_scope = [token.data.ident, token.data.type_.name]
            scopes.append(curr_scope)
        case TokenKind.UPSCOPE:
            scopes.pop()
            if scopes:
                curr_scope = scopes[-1]
        case TokenKind.VAR:
            # TODO: do we need the bit_index field ? (e.g. bit_index=(7,0) for MSB first wire)
            vars.append([curr_scope[0], " -> ".join([scope[0] for scope in scopes[:]]), token.data.type_.value, token.data.size, token.data.id_code, token.data.reference])
        case TokenKind.CHANGE_TIME:
            curr_timestamp = token.data
        case TokenKind.CHANGE_VECTOR:
            value_change.append([curr_timestamp, token.data.id_code, token.data.value])
        case TokenKind.CHANGE_SCALAR:
            value_change.append([curr_timestamp, token.data.id_code, token.data.value])



In [64]:
vars = pd.DataFrame(vars, columns=['Scope', 'Scope Chain', 'Type', 'Size', 'ID', 'Name'])

In [92]:
# this block will take a curr_ID, and will return the only unique scopes for such ID
# plan is to create a LUT for all the IDs using this code so we can easily pair
# IDs --> Possible Scopes
def get_scopes_for_id(curr_ID):
    df_for_curr_id = vars[vars.ID == curr_ID]
    scope_chain_curr_id = df_for_curr_id["Scope Chain"].to_list()
    sorted_scopes = sorted(scope_chain_curr_id, key=lambda s: s.count('->'))
    results = []
    for scope in sorted_scopes:
        if not any(scope != other and scope in other for other in sorted_scopes):
            results.append(scope)
    return results

In [67]:
# create the compressed col
vars['Scopes Compressed'] = vars['ID'].apply(get_scopes_for_id)

In [69]:
value_change = np.array(value_change, dtype=object)

In [240]:
# create numeric IDs instead of the textual
sig_idx, sig_id = pd.factorize(vars.ID)
vars["signal_idx"] = sig_idx
def idx2id(idx):
    return np.array(sig_id[idx])

In [255]:
# takes a value_change array and will return a power trace along with its signal ids array
# can take is input a filter list e.g. ['K"', '#'] to only get power for specific signals
# filter list can be generated using idx2id from the IDs in the vars dataframe
VALID_POWER_MODELS = {'Identity', 'HammingWeight'}
def get_trace(value_change, power_model, filter=None):
    if power_model not in VALID_POWER_MODELS:
        raise ValueError(f"Invalid power model: {power_model}")
    if filter is not None:
        filter_mask = np.isin(value_change[:, 1], filter)
        arr = np.array(value_change[filter_mask], dtype=object)
        t = arr[:, 0].astype(int)
        sid = arr[:, 1]                   # string IDs
        # TODO: handle non-numeric VCD values like 'x' and 'z'
        val = arr[:, 2].astype(int)
    else:
        arr = np.array(value_change, dtype=object)
        t = arr[:, 0].astype(int)
        sid = arr[:, 1]                   # string IDs
        # TODO: handle non-numeric VCD values like 'x' and 'z'
        val = arr[:, 2].astype(int)

    # Map IDs → column indices
    sig_idx, sig_labels = pd.factorize(sid)  # or np.unique with return_inverse

    num_timesteps = t.max() + 1
    num_signals = len(sig_labels)
    A = np.full((num_timesteps, num_signals), np.nan, dtype=float)

    # Write changes at their timestamps
    A[t, sig_idx] = val

    # Forward-fill down the time axis (last observation carried forward)
    mask = ~np.isnan(A)
    # indices of last seen row per column
    last = np.where(mask, np.arange(num_timesteps)[:, None], -1)
    np.maximum.accumulate(last, axis=0, out=last)
    last = last.clip(min=0)
    A_filled = A[last, np.arange(num_signals)]

    # If you want ints (and you know every series is defined from t=0)
    signal_names = sig_labels.tolist()
    if power_model == 'Identity':
        power_trace = A_filled.astype(int)
    if power_model == 'HammingWeight':
        power_trace = np.bitwise_count(A_filled.astype(int))
    return power_trace, signal_names

In [257]:
# this creates a granual hamming weight power trace for only signals with ids [1,2,3]
# returns the power trace and the mapping for the signals
granular_trace, mappings = get_trace(value_change, 'HammingWeight', idx2id([1,2,3]))
# actual power trace by summing
power_trace = granular_trace.sum(axis=1)